# Hafta 7 — Karar Ağaçları

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Karar ağacı, bir dizi evet/hayır sorusuyla sonuca ulaşır: "Hava güneşli mi? Evetse nem yüksek mi? Yüksekse bugün tenis oynanmaz." Bilgisayar her adımda verileri en iyi ayıran soruyu seçer. Bu yöntemin en büyük avantajı, sonucun herkesin kolayca okuyup anlayabileceği kurallar halinde ortaya çıkmasıdır.

Bu haftadan sonra ilk yedi haftanın konularını kapsayan ara sınav yapılır.

---

## Bu Haftanın Hedefleri

- Karar ağacının nasıl tahmin yaptığını anlamak
- Entropi, bilgi kazancı ve Gini değerlerini elle hesaplamak
- Karar ağacını sıfırdan kodlamak
- Aşırı öğrenmeyi budama ile önlemek

---

## 1. Karar Ağacı Nasıl Çalışır?

```
                 [Hava durumu?]
               ╱       │        ╲
        Güneşli     Bulutlu     Yağmurlu
           │           │           │
       [Nem?]       OYNA ✓     [Rüzgar?]
       ╱    ╲                   ╱     ╲
   Yüksek  Normal             Var     Yok
     │       │                 │       │
  OYNAMA   OYNA            OYNAMA    OYNA
```

Yeni bir örnek kökten başlar, her sorunun cevabına göre ilgili dala ilerler ve bir **yaprağa** ulaşınca oradaki kararı alır.

**Asıl soru:** Hangi soruyu önce sormalıyız? Cevap: Veriyi en **saf** (homojen) gruplara ayıran soruyu.

---

## 2. Saflığı Ölçmek

| Ölçü | Formül | [● ● ● ●] | [● ● ● ○] | [● ● ○ ○] |
|---|---|---|---|---|
| **Entropi** | −Σ p·log₂(p) | 0 | 0.81 | 1 (en karışık) |
| **Gini** | 1 − Σ p² | 0 | 0.375 | 0.5 (en karışık) |

Saf bir grupta iki ölçü de 0'dır; grup karıştıkça değerleri büyür.

**Bilgi kazancı:** Bir sorunun ne kadar iyi olduğunu ölçer.

```
Bilgi kazancı = Bölmeden önceki entropi − Bölmeden sonraki ağırlıklı ortalama entropi
```

Algoritma her adımda **bilgi kazancı en yüksek** soruyu seçer.

---

## 3. Elle Ağaç Kurmak: "Tenis Oynayalım mı?"

14 günlük kayıtta 9 gün oynanmış (Evet), 5 gün oynanmamış (Hayır).

**Başlangıç entropisi:** −(9/14)·log₂(9/14) − (5/14)·log₂(5/14) = **0.940**

**"Hava" sorusunun kazancı:**

| Hava | Evet / Hayır | Entropi |
|---|---|---|
| Güneşli (5 gün) | 2 / 3 | 0.971 |
| Bulutlu (4 gün) | 4 / 0 | 0 |
| Yağmurlu (5 gün) | 3 / 2 | 0.971 |

Ağırlıklı ortalama = (5/14)·0.971 + (4/14)·0 + (5/14)·0.971 = 0.694
Kazanç = 0.940 − 0.694 = **0.246**

Diğer sorular için aynı hesap yapılır:

| Soru | Bilgi kazancı |
|---|---|
| **Hava** | **0.246** ← en iyisi |
| Nem | 0.151 |
| Rüzgar | 0.048 |
| Sıcaklık | 0.029 |

Kök soru **Hava** olur. "Bulutlu" dalı tamamen saf olduğu için doğrudan "Oyna" yaprağına dönüşür. Güneşli dalda en iyi soru **Nem**, yağmurlu dalda ise **Rüzgar** çıkar. Bu iki soru dalları tamamen saf gruplara ayırır ve Bölüm 1'deki ağaç elde edilir.

### Sayısal öznitelikler

Yaş gibi sayısal özniteliklerde soru "yaş ≤ t?" biçimindedir. Değerler sıralanır, ardışık değerlerin orta noktaları aday eşik olarak denenir ve en saf bölmeyi veren eşik seçilir.

---

## 4. Sıfırdan Karar Ağacı

In [ ]:
import numpy as np

def gini(y):
    _, sayi = np.unique(y, return_counts=True)
    return 1 - np.sum((sayi / len(y)) ** 2)

def en_iyi_bolme(X, y):
    en_iyi = (0, None, None)                                  # (kazanç, öznitelik, eşik)
    for j in range(X.shape[1]):
        degerler = np.unique(X[:, j])
        for t in (degerler[:-1] + degerler[1:]) / 2:          # orta noktalar
            sol = X[:, j] <= t
            sonra = sol.mean() * gini(y[sol]) + (~sol).mean() * gini(y[~sol])
            if gini(y) - sonra > en_iyi[0]:
                en_iyi = (gini(y) - sonra, j, t)
    return en_iyi

def agac_kur(X, y, derinlik=0, max_derinlik=3):
    kazanc, j, t = en_iyi_bolme(X, y)
    if kazanc == 0 or derinlik == max_derinlik:               # durma koşulu → yaprak
        return int(np.bincount(y).argmax())
    sol = X[:, j] <= t
    return (int(j), float(t), agac_kur(X[sol], y[sol], derinlik + 1, max_derinlik),
                  agac_kur(X[~sol], y[~sol], derinlik + 1, max_derinlik))

def tahmin(agac, x):
    while isinstance(agac, tuple):
        j, t, sol, sag = agac
        agac = sol if x[j] <= t else sag
    return agac

# Örnek: yaşa göre kredi onayı
X = np.array([[22], [25], [30], [35], [40], [52]])
y = np.array([0, 0, 1, 1, 1, 0])
agac = agac_kur(X, y)
print(agac)                                   # (0, 27.5, 0, (0, 46.0, 1, 0))
print([tahmin(agac, x) for x in X])           # [0, 0, 1, 1, 1, 0]

Çıktı şu kuralı anlatır: Yaş ≤ 27.5 ise reddet; değilse yaş ≤ 46 ise onayla, büyükse reddet.

---

## 5. Aşırı Öğrenme ve Budama

Sınırsız büyüyen bir ağaç, her yaprakta tek bir örnek kalana kadar bölünür. Eğitim doğruluğu %100 olur, ama ağaç gürültüyü ezberlemiştir.

| Yöntem | Nasıl? |
|---|---|
| **Ön budama** | Ağacı büyürken sınırla: `max_depth`, `min_samples_leaf` |
| **Son budama** | Önce tam ağacı büyüt, sonra faydasız dalları kes: `ccp_alpha` |

En iyi değerler çapraz doğrulamayla seçilir.

| Güçlü yönler | Zayıf yönler |
|---|---|
| Kurallar okunabilir ve açıklanabilir | Kolayca aşırı öğrenir |
| Ölçekleme gerektirmez | Veri biraz değişince ağaç çok değişir |
| Doğrusal olmayan ilişkileri yakalar | Sınırları eksenlere paralel, merdiven biçimlidir |

---

## 6. Laboratuvar: Ağaç Görselleştirme

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score

iris = load_iris()
model = DecisionTreeClassifier(max_depth=3, random_state=0).fit(iris.data, iris.target)
plt.figure(figsize=(14, 7))
plot_tree(model, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.show()

# Derinliğin etkisi
X, y = load_breast_cancer(return_X_y=True)
for d in [1, 2, 3, 5, 10, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=0)
    print(f"max_depth={d}: eğitim {m.fit(X, y).score(X, y):.3f}  CV {cross_val_score(m, X, y, cv=5).mean():.3f}")

**Görevler:**
1. Ağaç çizimindeki kök düğümün Gini değerini elle doğrulayın.
2. Derinlik arttıkça eğitim ve CV doğruluğu nasıl değişiyor? Aşırı öğrenme hangi derinlikte başlıyor?

### Proje önerisi teslimi

Bu hafta dönem projesi önerinizi teslim ediyorsunuz. Öneride şunlar bulunmalıdır: problem tanımı, veri seti ve kaynağı, problem türü, değerlendirme metriği, karşılaştırma için basit bir başlangıç modeli (baseline) ve planlanan yöntemler.

---

## 7. Özet

- Karar ağacı veriyi ardışık evet/hayır sorularıyla saf gruplara böler.
- Entropi ve Gini saflığı ölçer; en yüksek bilgi kazancını veren soru seçilir.
- Sınırsız ağaçlar aşırı öğrenir; derinlik sınırı ve budama ile kontrol edilir.

## Kendinizi Sınayın

1. [● ● ● ● ○ ○] grubunun Gini değerini hesaplayın.
2. Karar ağaçları neden ölçeklemeden etkilenmez?
3. Her satırda farklı olan "müşteri numarası" sütunu neden en yüksek bilgi kazancını verir ve bu neden bir sorundur?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.